# Homework 1 - Deep Learning Winter 2024

Student 1: Rajaa Haj 322512690

Student 2: Aseel Shaheen 212393532



In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [58]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [59]:
import pandas as pd

# Define the paths to your datasets
train_path = '/content/drive/MyDrive/HW-1/part1_train.csv'
test_path = '/content/drive/MyDrive/HW-1/part1_test.csv'

# Load the datasets
train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

# Display the first few rows of the training dataset
# print(train_data.head())
# print(test_data.head())


# **In order to scale the data we need first to covnert non-numerical values into numerical represtnation. so what we do is we apply one-hot encoding in order to represnt them.**

In [96]:

# Identify categorical columns that exist in both train and test datasets
categorical_columns = list(set(train_data.select_dtypes(include=['object']).columns) &
                           set(test_data.select_dtypes(include=['object']).columns))

# Extract target variable before encoding (assuming last column is the target)
y_train = train_data.iloc[:, -1]
y_test = test_data.iloc[:, -1]

#need to remove the '.' in order to make the data clear and now convert it to numerical.
y_test = test_data.iloc[:, -1].str.strip().str.replace('.', '', regex=False)
# Strip whitespace from target column values
y_train = y_train.str.strip()
y_test = y_test.str.strip()


# Define the mapping for target values
label_mapping = {"<=50K": 0, ">50K": 1}

# Apply the mapping
y_train = y_train.map(label_mapping)
y_test = y_test.map(label_mapping)


# Apply one-hot encoding separately (for training and test (without target))
train_data_encoded = pd.get_dummies(train_data, columns=categorical_columns)
test_data_encoded = pd.get_dummies(test_data, columns=categorical_columns)

# Align columns between train and test datasets to handle mismatched categories
train_data_aligned, test_data_aligned = train_data_encoded.align(test_data_encoded, join='inner', axis=1)


# **Exctract features and exclude the target for training and test and scaling the data using the StandardScaler.**

In [97]:

# Extract features (after alignment)
X_train = train_data_aligned.drop(columns=train_data_aligned.columns[-1])  # Drop target column
X_test = test_data_aligned.drop(columns=test_data_aligned.columns[-1])    # Drop target column

# Normalize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# # Print shapes to verify
# print("X_train_scaled shape:", X_train_scaled.shape)
# print("X_test_scaled shape:", X_test_scaled.shape)
# print("y_train shape:", y_train.shape)
# print("y_test shape:", y_test.shape)


In [98]:
# Convert data to PyTorch tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

Now we Print the distribution of yearly income for the individuals and the
percentage of them more / less than 50k/year

In [100]:
# Compute distribution for y_train
unique, counts = torch.unique(y_train, return_counts=True)

# Print the distribution
print("Income Distribution in Training Data:")
for value, count in zip(unique, counts):
    label = "<=50K" if value.item() == 0 else ">50K"
    print(f"{label}: {count.item()}")

# Compute percentages
total = counts.sum().item()
percentages = (counts.float() / total) * 100

# Print the percentages
print("\nIncome Percentage in Training Data:")
for value, percent in zip(unique, percentages):
    label = "<=50K" if value.item() == 0 else ">50K"
    print(f"{label}: {percent.item():.2f}%")


Income Distribution in Training Data:
<=50K: 24719
>50K: 7841

Income Percentage in Training Data:
<=50K: 75.92%
>50K: 24.08%


In [124]:
import torch.nn as nn
import torch.nn.functional as F

class FeedForwardNN(nn.Module):
    def __init__(self, input_size):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)  # Hidden layer 1
        self.dropout1 = nn.Dropout(p=0.2)    # Dropout for regularization
        self.fc2 = nn.Linear(64, 32)         # Hidden layer 2
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(32, 2)          # Output layer

    def forward(self, x):
        x = F.relu(self.fc1(x))  # Activation function for first layer
        x = F.relu(self.fc2(x))  # Activation function for second layer
        x = self.fc3(x)          # No activation for output layer (raw scores)
        return x


In [126]:
from torch.utils.data import TensorDataset, DataLoader

# Input size is the number of features (but its gone to 51 due to one-code being applied which increased it)
input_size = X_train.shape[1]

# Define the model and hyperamreters
model = FeedForwardNN(input_size)
learning_rate = 0.001
epochs = 20
batch_size = 32


# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
#we make each epoch to go on the batch size and not the entire data
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
with torch.no_grad():
    test_outputs = model(X_test)
    _, test_predictions = torch.max(test_outputs, 1)

accuracy = (test_predictions == y_test).sum().item() / y_test.size(0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")




Epoch 1/20, Loss: 0.4290
Epoch 2/20, Loss: 0.4170
Epoch 3/20, Loss: 0.4145
Epoch 4/20, Loss: 0.4133
Epoch 5/20, Loss: 0.4120
Epoch 6/20, Loss: 0.4118
Epoch 7/20, Loss: 0.4113
Epoch 8/20, Loss: 0.4107
Epoch 9/20, Loss: 0.4101
Epoch 10/20, Loss: 0.4103
Epoch 11/20, Loss: 0.4097
Epoch 12/20, Loss: 0.4096
Epoch 13/20, Loss: 0.4096
Epoch 14/20, Loss: 0.4091
Epoch 15/20, Loss: 0.4085
Epoch 16/20, Loss: 0.4088
Epoch 17/20, Loss: 0.4080
Epoch 18/20, Loss: 0.4082
Epoch 19/20, Loss: 0.4078
Epoch 20/20, Loss: 0.4082
Test Accuracy: 79.44%
